# German Credit Data — KNN for loan default prediction

**Problem:** Predict whether a customer is a **bad credit risk** (likely to default) vs **good credit risk**, using the Statlog German Credit dataset.

**Algorithm:** k-Nearest Neighbors (supervised classification). Feature scaling is required because KNN uses distance metrics.

**Data citation:** Dua, D. and Graff, C. (2019). *UCI Machine Learning Repository*. Irvine, CA: University of California, School of Information and Computer Sciences. Dataset: [Statlog (German Credit Data)](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data). Local copies: `german.data` (mixed types), `german.data-numeric` (all numeric, used here). See `german.doc` for attribute descriptions and the **cost matrix** (misclassifying a bad customer as good is penalized more heavily).

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Folder containing german.data-numeric (same directory as this notebook)
DATA_DIR = Path(".")
NUMERIC_FILE = DATA_DIR / "german.data-numeric"

In [10]:
# Load: 1000 rows, whitespace-separated, no header
# Last column: 1 = good credit, 2 = bad credit (default / high risk)
df = pd.read_csv(NUMERIC_FILE, sep=r"\s+", header=None, engine="python")
assert df.shape[0] == 1000, f"Expected 1000 rows, got {df.shape[0]}"

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Binary labels for reporting: 0 = good (no default), 1 = default / bad risk
y_binary = (y == 2).astype(int)

print("Shape X:", X.shape, "| Classes (raw):", np.unique(y))
print("Class counts (raw 1=good, 2=bad):\n", pd.Series(y).value_counts().sort_index())
print("Default rate (bad=1):", y_binary.mean().round(3))

Shape X: (1000, 24) | Classes (raw): [1 2]
Class counts (raw 1=good, 2=bad):
 1    700
2    300
Name: count, dtype: int64
Default rate (bad=1): 0.3


In [11]:
# Stratified split preserves ~30% bad cases in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.3,
    random_state=42,
    stratify=y_binary,
)

# Pipeline: scale all features then KNN (critical for distance-based model)
knn_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5, weights="uniform")),
    ]
)

knn_pipe.fit(X_train, y_train)
y_pred = knn_pipe.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1 (default=positive class):", round(f1_score(y_test, y_pred, pos_label=1), 4))
print("\nConfusion matrix [ [TN, FP], [FN, TP] ] — rows=true, cols=predicted")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report (0=good, 1=default):")
print(classification_report(y_test, y_pred, target_names=["good", "default"]))

Accuracy: 0.6867
F1 (default=positive class): 0.3562

Confusion matrix [ [TN, FP], [FN, TP] ] — rows=true, cols=predicted
[[180  30]
 [ 64  26]]

Classification report (0=good, 1=default):
              precision    recall  f1-score   support

        good       0.74      0.86      0.79       210
     default       0.46      0.29      0.36        90

    accuracy                           0.69       300
   macro avg       0.60      0.57      0.57       300
weighted avg       0.66      0.69      0.66       300



In [13]:
# Optional: tune k with cross-validation (same pipeline)
param_grid = {"knn__n_neighbors": np.arange(1, 31, 2), "knn__weights": ["uniform", "distance"]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    knn_pipe,
    param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV F1 (default):", round(grid.best_score_, 4))

y_pred_tuned = grid.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, y_pred_tuned), 4))
print("Test F1 (default):", round(f1_score(y_test, y_pred_tuned, pos_label=1), 4))

Best params: {'knn__n_neighbors': np.int64(1), 'knn__weights': 'uniform'}
Best CV F1 (default): 0.4455
Test accuracy: 0.67
Test F1 (default): 0.4649
